In [0]:
%sql
create connection if not exists earth_quake_conn
type HTTP
options (
  host = 'https://earthquake.usgs.gov',
  port = 443,
  bearer_token = 'na',
  base_path = '/earthquakes/feed/v1.0/'

)

In [0]:
dbutils.widgets.text('catalog_name','prod_catalog','dev_catalog')
catalog_name = dbutils.widgets.get('catalog_name')

In [0]:
%py

spark.sql(
    f" use catalog {catalog_name}"
)
spark.sql(
    "use schema bronze"
    )
spark.sql(
    "create volume if not exists earthquake_data"
)    


In [0]:

%py
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
conn = w.connections.get("earth_quake_conn")
base_url = f"{conn.options['host']}{conn.options['base_path']}"
# print(base_url)
# print(conn)

In [0]:
%sql
SHOW VOLUMES

In [0]:
import requests
import json 
import datetime
url  = f'{base_url}summary/all_day.geojson'

response = requests.get(url)
if(response.status_code != 200):
  raise Exception(f"Error in getting response data from {url}")
data = response.json()
current_date = datetime.datetime.now().strftime("%Y-%m-%d")
dbutils.fs.put(f'/Volumes/{catalog_name}/bronze/earthquake_data/earthquake_data_{current_date}.geo.json', json.dumps(data), overwrite=True)